# YOLO для кейсу 4.5: навчання на Kaggle

**Перед запуском:** праворуч Settings → Accelerator **GPU T4 x2**, Internet **On**. У клітинці ДАНІ вже стоять посилання: сирий Anti-UAV410 і пакет команди AntiUAV410_SKY (маніфест відібраних кадрів неба). Додається синтетичне сонце.

**Запуск:** змінити `MODEL` у клітинці нижче → **Save Version → Save & Run All**. Ноутбук працює у фоні, вкладку можна закрити. Результат у вкладці Output: `best.pt`, `best.onnx`, `metrics.json`, `results.csv`.

Хто що запускає: див. README_TEAM.md.

In [ ]:
# ======= ПАРАМЕТРИ: міняти лише тут =======
MODEL = 'yolo26m-p2'   # yolo26m | yolo26m-p2 | yolo26l-p2 | yolo26x-p2
EPOCHS = 50            # час однаково обмежений HOURS
BATCH = 16             # на 2 GPU разом
HOURS = 10.5           # страховка від ліміту Kaggle 12 год: навчання саме зупиниться і збереже best.pt
GPUS = [0, 1]          # якщо впаде з помилкою DDP, поставити [0]


In [ ]:
import os; os.makedirs(os.path.dirname('/kaggle/working/prepare_yolo.py'), exist_ok=True)
open('/kaggle/working/prepare_yolo.py', 'w').write(r'''"""Anti-UAV410 (+ відібрані негативи) -> YOLO-датасет і ті самі кадри для DNA-Net (маски з рамок).

  python prepare_yolo.py --root Anti-UAV-410 --neg negatives --out auav_yolo --stride 10 --neg_frac 0.15 --dnanet

Anti-UAV410: train/val/test/<серія>/*.jpg + IR_label.json {"exist": [...], "gt_rect": [[x, y, w, h], ...]}.
- Спліт офіційний, по серіях: серії не перетинаються між train, val і test.
- Сусідні кадри майже однакові, тому беремо кожен stride-й кадр із ціллю.
- Негативи: кадри без цілі з папки --neg/<split>, їх частка neg_frac від усіх кадрів спліту.
- DNA-Net вчиться на тих самих кадрах: маска = пікселі в рамці, що відрізняються від фону (Otsu), інакше еліпс.
"""
import os, glob, json, argparse, random
import numpy as np, cv2


def clip_box(box, W, H):
    """Рамку, що виходить за край кадру, обрізаємо по кадру. None, якщо від неї лишилось менше 1 px."""
    x, y, w, h = box
    x0, y0, x1, y1 = max(x, 0.0), max(y, 0.0), min(x + w, W), min(y + h, H)
    return (x0, y0, x1 - x0, y1 - y0) if x1 - x0 >= 1 and y1 - y0 >= 1 else None


def box_to_mask(img, boxes):
    H, W = img.shape
    m = np.zeros(img.shape, np.uint8)
    for b in boxes:
        b = clip_box(b, W, H)
        if b is None: continue
        x, y, w, h = b
        x0, y0 = int(x), int(y); x1, y1 = min(int(np.ceil(x + w)), W), min(int(np.ceil(y + h)), H)
        if x1 - x0 < 2 or y1 - y0 < 2: continue
        ell = np.zeros((y1 - y0, x1 - x0), np.uint8)
        cv2.ellipse(ell, ((x1 - x0) // 2, (y1 - y0) // 2), ((x1 - x0) // 2, (y1 - y0) // 2), 0, 0, 360, 1, -1)
        p = int(max(w, h) // 2 + 2)
        bg = np.median(img[max(y0 - p, 0):y1 + p, max(x0 - p, 0):x1 + p])
        diff = np.abs(img[y0:y1, x0:x1].astype(np.float32) - bg)
        _, th = cv2.threshold(cv2.normalize(diff, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8), 0, 1, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        sel = th & ell
        if diff.max() < 4 or sel.sum() < 0.2 * ell.sum(): sel = ell
        m[y0:y1, x0:x1] |= sel * 255
    return m


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", required=True); ap.add_argument("--neg"); ap.add_argument("--out", required=True)
    ap.add_argument("--stride", type=int, default=10); ap.add_argument("--neg_frac", type=float, default=0.15)
    ap.add_argument("--dnanet", action="store_true"); ap.add_argument("--seed", type=int, default=0)
    a = ap.parse_args()
    rng = random.Random(a.seed); stats = {}
    for split in ["train", "val", "test"]:
        for d in ["images", "labels"] + (["masks"] if a.dnanet else []):
            os.makedirs(f"{a.out}/{d}/{split}", exist_ok=True)
        names, sizes = [], []
        for sd in sorted(glob.glob(f"{a.root}/{split}/*/")):
            seq = os.path.basename(sd.rstrip("/"))
            lf = next((f for f in ["IR_label.json", "infrared.json"] if os.path.exists(sd + f)), None)
            if lf is None: continue
            lab = json.load(open(sd + lf)); frames = sorted(glob.glob(sd + "*.jpg"))
            for i in range(0, min(len(frames), len(lab["exist"])), a.stride):
                box = lab["gt_rect"][i]
                if not lab["exist"][i] or len(box) != 4 or box[2] < 1 or box[3] < 1: continue
                img = cv2.imread(frames[i], 0); H, W = img.shape
                box = clip_box(box, W, H)                       # частина рамок Anti-UAV410 виходить за край кадру
                if box is None: continue
                x, y, w, h = box; n = f"{seq}_{i:06d}"
                cv2.imwrite(f"{a.out}/images/{split}/{n}.jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 95])
                open(f"{a.out}/labels/{split}/{n}.txt", "w").write(f"0 {(x + w / 2) / W:.6f} {(y + h / 2) / H:.6f} {w / W:.6f} {h / H:.6f}\n")
                if a.dnanet: cv2.imwrite(f"{a.out}/masks/{split}/{n}.png", box_to_mask(img, [box]))
                names.append(n); sizes.append(max(w, h))
        n_pos = len(names)
        negs = sorted(glob.glob(f"{a.neg}/{split}/*.jpg")) if a.neg else []
        k = min(len(negs), round(a.neg_frac / (1 - a.neg_frac) * n_pos)); rng.shuffle(negs)
        for p in negs[:k]:
            n = "neg_" + os.path.basename(p)[:-4]; img = cv2.imread(p, 0)
            cv2.imwrite(f"{a.out}/images/{split}/{n}.jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 95])
            open(f"{a.out}/labels/{split}/{n}.txt", "w").close()
            if a.dnanet: cv2.imwrite(f"{a.out}/masks/{split}/{n}.png", np.zeros_like(img))
            names.append(n)
        s = np.array(sizes)
        stats[split] = dict(positives=n_pos, negatives=k, neg_share=round(k / max(len(names), 1), 3),
                            box_lt5=round(float((s < 5).mean()), 3) if len(s) else None,
                            box_5_50=round(float(((s >= 5) & (s <= 50)).mean()), 3) if len(s) else None,
                            box_gt50=round(float((s > 50).mean()), 3) if len(s) else None)
        if a.dnanet:
            os.makedirs(f"{a.out}/img_idx", exist_ok=True)
            open(f"{a.out}/img_idx/{split}.txt", "w").write("\n".join(f"{split}/{n}" for n in names))
        print(split, stats[split], flush=True)
    open(f"{a.out}/data.yaml", "w").write(f"path: {os.path.abspath(a.out)}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n  0: target\n")
    json.dump(stats, open(f"{a.out}/stats.json", "w"), indent=1)


if __name__ == "__main__":
    main()
'''); print('записано', '/kaggle/working/prepare_yolo.py')

In [ ]:
import os; os.makedirs(os.path.dirname('/kaggle/working/run_yolo.py'), exist_ok=True)
open('/kaggle/working/run_yolo.py', 'w').write(r'''"""YOLO для кейсу 4.5: швидкий перебір моделей, повне навчання, TensorRT FP16/INT8, оцінка за ТЗ.

  python run_yolo.py sweep  --data auav_sweep/data.yaml                     # кроки 2-3: яка модель і голова (P3 чи P2)
                            # auav_sweep = prepare_yolo.py --stride 30 (у 3 рази менше кадрів, ті самі серії)
  python run_yolo.py train  --data auav_yolo/data.yaml --model yolo26l-p2   # крок 5
  python run_yolo.py export --weights runs/.../best.pt --data auav_yolo/data.yaml   # кроки 6-7: engine FP16 і INT8
  python run_yolo.py eval   --weights best.engine --val auav_yolo:val --test auav_yolo:test sky=testsets/sky_targets ...

Оцінка така сама, як у skydet: влучання = центр рамки в рамці цілі +-3 px, цілі <5 і >50 px ігноруються,
FP рахується на кожному наборі окремо, поріг conf підбирається на val (FP <= fp_budget), FPS = увесь predict на кадрі 640x512.
"""
import os, sys, json, time, glob, argparse
import numpy as np, cv2

TOL = 3


def gts_yolo(lbl, W=640, H=512):
    out = []
    if os.path.exists(lbl):
        for l in open(lbl).read().split("\n"):
            if l.strip():
                _, cx, cy, w, h = map(float, l.split()); w, h = w * W, h * H
                out.append((cx * W - w / 2, cy * H - h / 2, w, h))
    return out


def gts_mask(m):
    n, _, st, _ = cv2.connectedComponentsWithStats((m > 127).astype(np.uint8), 8)
    return [tuple(map(float, st[i, :4])) for i in range(1, n)]


def load_set(spec):
    """'dir:split' = YOLO-датасет (images/split, labels/split); 'name=dir' = набір skydet (images/, masks/, list.txt)."""
    if "=" in spec:
        name, d = spec.split("=", 1)
        items = [(f"{d}/images/{n}.png", gts_mask(cv2.imread(f"{d}/masks/{n}.png", 0))) for n in open(f"{d}/list.txt").read().split()]
    else:
        d, split = spec.rsplit(":", 1); name = split
        items = [(p, gts_yolo(f"{d}/labels/{split}/{os.path.basename(p).rsplit('.', 1)[0]}.txt"))
                 for p in sorted(glob.glob(f"{d}/images/{split}/*"))]
    return name, items


def predict_all(model, items, imgsz):
    out = []
    for p, g in items:
        r = model.predict(cv2.imread(p), imgsz=imgsz, conf=0.01, verbose=False)[0].boxes
        out.append((r.xywh.cpu().numpy(), r.conf.cpu().numpy(), g))
    return out


def score(preds, conf, topk):
    tp = n_ok = 0; fps = []
    for xywh, c, gts in preds:
        keep = np.argsort(-c)[: topk or None]; keep = [i for i in keep if c[i] >= conf]
        hit, fp = set(), 0
        for i in keep:
            cx, cy = xywh[i, 0], xywh[i, 1]
            h = [j for j, (x, y, w, hh) in enumerate(gts) if x - TOL <= cx <= x + w + TOL and y - TOL <= cy <= y + hh + TOL]
            fp += not h
            hit.update(j for j in h if min(gts[j][2], gts[j][3]) >= 5 and max(gts[j][2], gts[j][3]) <= 50)
        tp += len(hit); n_ok += sum(min(w, h) >= 5 and max(w, h) <= 50 for _, _, w, h in gts); fps.append(fp)
    fps = np.array(fps)
    return dict(frames=len(preds), targets=int(n_ok), recall=round(tp / n_ok, 3) if n_ok else None,
                fp_per_frame=round(float(fps.mean()), 3), fp_max=int(fps.max()), frames_fp_gt1=round(float((fps > 1).mean()), 3))


def speed(model, imgsz, n=200):
    img = np.full((512, 640, 3), 90, np.uint8); cv2.circle(img, (300, 200), 4, (220,) * 3, -1)
    for _ in range(20): model.predict(img, imgsz=imgsz, verbose=False)
    t = []
    for _ in range(n):
        t0 = time.perf_counter(); model.predict(img, imgsz=imgsz, verbose=False); t.append((time.perf_counter() - t0) * 1000)
    return dict(ms_mean=round(float(np.mean(t)), 2), ms_p95=round(float(np.percentile(t, 95)), 2), fps=round(1000 / float(np.mean(t)), 1))


AUG = dict(hsv_h=0, hsv_s=0, hsv_v=0.3, degrees=0, scale=0.3, mosaic=1.0, close_mosaic=10, fliplr=0.5, mixup=0)


def build(name):
    """Повертає (модель, ваги для старту). P2-голова не має готових ваг: будуємо з yaml і стартуємо
    з ваг COCO звичайної моделі того ж розміру (параметр pretrained у train, працює і з 2 GPU)."""
    from ultralytics import YOLO
    if name.endswith(".pt") or name.endswith(".engine"):
        return YOLO(name), True
    if "-p2" in name:
        return YOLO(name + ".yaml"), name.replace("-p2", "") + ".pt"
    return YOLO(name + ".pt"), True


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("cmd", choices=["sweep", "train", "export", "eval"])
    ap.add_argument("--data"); ap.add_argument("--model", default="yolo26l-p2"); ap.add_argument("--weights")
    ap.add_argument("--models", default="yolo26l,yolo26s-p2,yolo26m-p2,yolo26l-p2,yolo26x-p2")
    ap.add_argument("--epochs", type=int, default=80); ap.add_argument("--imgsz", type=int, default=640)
    ap.add_argument("--batch", type=int, default=16); ap.add_argument("--fraction", type=float, default=1.0)
    ap.add_argument("--val", nargs="*", default=[]); ap.add_argument("--test", nargs="*", default=[])
    ap.add_argument("--fp_budget", type=float, default=0.7); ap.add_argument("--json", default="yolo_results.json")
    a = ap.parse_args()
    import torch
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"

    if a.cmd == "sweep":                  # коротке навчання на частині даних, щоб порівняти моделі між собою
        rows = []
        for name in a.models.split(","):
            m, pre = build(name)
            m.train(data=a.data, pretrained=pre, epochs=15, imgsz=a.imgsz, batch=a.batch, project="sweep", name=name,
                    exist_ok=True, verbose=False, plots=False, **AUG)
            v = m.val(data=a.data, split="val", imgsz=a.imgsz, verbose=False)
            best, _ = build(f"sweep/{name}/weights/best.pt")
            rows.append(dict(model=name, params_M=round(sum(p.numel() for p in best.model.parameters()) / 1e6, 1),
                             mAP50=round(float(v.box.map50), 3), recall=round(float(v.box.mr), 3), gpu=gpu,
                             **{f"pytorch_{k}": x for k, x in speed(best, [512, 640]).items()}))
            print(rows[-1], flush=True); json.dump(rows, open("sweep.json", "w"), indent=1)

    elif a.cmd == "train":
        m, pre = build(a.model)
        m.train(data=a.data, pretrained=pre, epochs=a.epochs, imgsz=a.imgsz, batch=a.batch, fraction=a.fraction, project="train", name=a.model,
                exist_ok=True, patience=20, cos_lr=True, **AUG)

    elif a.cmd == "export":               # крок 6: TensorRT FP16; крок 7: INT8 з калібруванням на train
        from ultralytics import YOLO
        for kw in [dict(half=True), dict(int8=True, data=a.data, fraction=0.1)]:
            p = YOLO(a.weights).export(format="engine", imgsz=[512, 640], batch=1, **kw)
            os.replace(p, p.replace(".engine", "_int8.engine" if "int8" in kw else "_fp16.engine")); print("saved", p)

    elif a.cmd == "eval":
        from ultralytics import YOLO
        m = YOLO(a.weights, task="detect")
        val = [(n, predict_all(m, it, [512, 640])) for n, it in map(load_set, a.val)]
        best = None
        for conf in [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5, 0.6]:
            for k in [1, 3, 5]:
                r = {n: score(p, conf, k) for n, p in val}
                ok = all(x["fp_per_frame"] <= a.fp_budget for x in r.values())
                rec = np.mean([x["recall"] for x in r.values() if x["recall"] is not None])
                if ok and (best is None or rec > best[0]): best = (rec, conf, k, r)
        if best is None:                  # жоден поріг не вклався в бюджет FP: беремо найсуворіший
            best = (0, 0.6, 1, {n: score(p, 0.6, 1) for n, p in val}); print("УВАГА: FP > fp_budget навіть при conf 0.6, top-1")
        _, conf, k, rv = best
        res = dict(weights=a.weights, gpu=gpu, conf=conf, topk=k, val=rv,
                   test={n: score(predict_all(m, it, [512, 640]), conf, k) for n, it in map(load_set, a.test)}, speed=speed(m, [512, 640]))
        print(json.dumps(res, indent=1, ensure_ascii=False)); json.dump(res, open(a.json, "w"), indent=1, ensure_ascii=False)


if __name__ == "__main__":
    main()
'''); print('записано', '/kaggle/working/run_yolo.py')

In [ ]:
# ======= ПАКЕТ КОМАНДИ AntiUAV410_SKY: builder і маніфест відібраних кадрів (вже проріджений під цей ноутбук) =======
import os, base64
os.makedirs('/tmp/dl/team_package', exist_ok=True)
open('/tmp/dl/team_package/build_antiuav_yolo.py', 'w').write(r'''#!/usr/bin/env python3
"""Build a single-class 640x512 YOLO dataset from a curated manifest and RAW ZIP.

Only selected image bytes are read. Inputs are read-only. Nonempty output directories
are refused. Exit codes: 0 complete; 1 partial build with row errors; 2 fatal input error.
"""
from __future__ import annotations
import argparse
import csv
import hashlib
import io
import json
import math
import re
import sys
import zipfile
from collections import Counter, defaultdict
from pathlib import Path, PurePosixPath

import pandas as pd
from PIL import Image
import yaml
from tqdm import tqdm

VERSION = '1.1.0'
SPLITS = ('train','val','test')
ROLES = ('positive','negative','hard_negative')
REQUIRED = {'sequence','frame_number','image_name','exist','valid_bbox','bbox_x','bbox_y','bbox_w','bbox_h','sample_role','assigned_split'}
ERROR_FIELDS = ['row','sequence','frame_number','image_name','error','detail']

def digest(path: Path) -> str:
    h=hashlib.sha256()
    with path.open('rb') as src:
        for block in iter(lambda:src.read(1024*1024),b''):h.update(block)
    return h.hexdigest()

def boolean(value) -> bool:
    text=str(value).strip().lower()
    if text in ('true','1','1.0'):return True
    if text in ('false','0','0.0'):return False
    raise ValueError(f'Invalid boolean {value!r}')

def output_stem(sequence: str, frame: int) -> str:
    safe=re.sub(r'[^A-Za-z0-9_-]','_',sequence)[:80]
    key=hashlib.sha256(sequence.encode('utf-8')).hexdigest()[:16]
    return f'{safe}__{key}__{frame:08d}'

def load_manifest(path: Path) -> pd.DataFrame:
    df=pd.read_csv(path,dtype={'sequence':str,'image_name':str,'sample_role':str,'assigned_split':str})
    missing=REQUIRED-set(df.columns)
    if missing:raise ValueError(f'Manifest missing columns: {sorted(missing)}')
    if df.empty:raise ValueError('Manifest is empty')
    for col in ['sequence','image_name','sample_role','assigned_split']:
        if df[col].isna().any() or df[col].str.strip().eq('').any():raise ValueError(f'Blank {col}')
    if not set(df.assigned_split)<=set(SPLITS):raise ValueError('Only train/val/test splits are allowed')
    if not set(df.sample_role)<=set(ROLES):raise ValueError('Unknown sample_role')
    for col in ['frame_number','exist']:
        vals=pd.to_numeric(df[col],errors='raise')
        if vals.isna().any() or not vals.map(lambda x:math.isfinite(x) and x==int(x)).all():raise ValueError(f'Invalid integer {col}')
        df[col]=vals.astype('int64')
    if (df.frame_number<1).any() or not set(df.exist)<={0,1}:raise ValueError('Invalid frame_number or exist')
    if df.duplicated(['sequence','frame_number']).any() or df.duplicated(['sequence','image_name']).any():raise ValueError('Duplicate frame keys')
    if (df.groupby('sequence').assigned_split.nunique()>1).any():raise ValueError('Sequence appears in multiple splits')
    if 'recording_group' in df:
        if df.recording_group.isna().any() or df.recording_group.astype(str).str.strip().eq('').any():raise ValueError('Blank recording_group')
        if (df.groupby('recording_group').assigned_split.nunique()>1).any():raise ValueError('Recording group appears in multiple splits')
    for row in df.itertuples():
        if '/' in row.sequence or '\\' in row.sequence or row.sequence in ('.','..'):raise ValueError('Sequence must be a directory component')
        if PurePosixPath(row.image_name).name!=row.image_name or '\\' in row.image_name or not row.image_name.lower().endswith(('.jpg','.jpeg')):raise ValueError('image_name must be a JPEG basename')
    df['valid_bbox']=df.valid_bbox.map(boolean)
    return df.sort_values(['assigned_split','sequence','frame_number']).reset_index(drop=True)

def index_zip(z: zipfile.ZipFile, sequences: set[str]):
    index=defaultdict(list)
    # ZIP central-directory metadata only. No unselected image is opened/extracted.
    for info in tqdm(z.infolist(),desc='Index ZIP',unit='entries'):
        if info.is_dir():continue
        parts=PurePosixPath(info.filename.replace('\\','/')).parts
        if not parts or not parts[-1].lower().endswith(('.jpg','.jpeg')):continue
        hits=sequences.intersection(parts[:-1])
        for seq in hits:index[(seq,parts[-1].lower())].append((info,parts))
    return index

def resolve_member(index, row):
    candidates=index.get((row.sequence,row.image_name.lower()),[])
    if not candidates:raise FileNotFoundError('Selected JPG not found under sequence directory')
    thermal=[c for c in candidates if any(p.lower() in {'infrared','ir','thermal'} for p in c[1][:-1])]
    if thermal:candidates=thermal
    else:
        candidates=[c for c in candidates if not any(p.lower() in {'visible','rgb','color'} for p in c[1][:-1])]
    if len(candidates)>1 and hasattr(row,'original_split'):
        same=[c for c in candidates if str(row.original_split).lower() in [p.lower() for p in c[1][:-1]]]
        if same:candidates=same
    if len(candidates)!=1:raise ValueError(f'Ambiguous or non-thermal JPG match: {len(candidates)} candidates')
    return candidates[0][0]

def make_label(row, width: int, height: int) -> str:
    if row.sample_role in ('negative','hard_negative'):
        if row.exist!=0 or row.valid_bbox:raise ValueError('Negative role contradicts exist/valid_bbox')
        return ''
    if row.exist!=1 or not row.valid_bbox:raise ValueError('Invalid positive bbox or contradictory exist')
    x,y,w,h=(float(getattr(row,f'bbox_{k}')) for k in ('x','y','w','h'))
    if not all(math.isfinite(v) for v in (x,y,w,h)):raise ValueError('Nonfinite bbox')
    if w<=0 or h<=0 or x<0 or y<0 or x+w>width or y+h>height:raise ValueError(f'BBox outside {width}x{height}: {(x,y,w,h)}')
    norm=((x+w/2)/width,(y+h/2)/height,w/width,h/height)
    if not all(0<=v<=1 for v in norm):raise ValueError('Normalized bbox outside [0,1]')
    return '0 '+' '.join(f'{v:.10f}' for v in norm)+'\n'

def preflight(source, manifest, df, out) -> int:
    """Validate every selected reference using ZIP metadata; no JPEG payloads read."""
    errors=[]; seen=set(); stems=set(); image_bytes=0
    try:
        with zipfile.ZipFile(source,'r') as z:
            index=index_zip(z,set(df.sequence))
            for number,row in enumerate(tqdm(df.itertuples(index=False),total=len(df),desc='Check references',unit='frames'),start=2):
                try:
                    make_label(row,640,512)
                    info=resolve_member(index,row)
                    if info.header_offset in seen:raise ValueError('Same RAW member referenced twice')
                    stem=output_stem(row.sequence,row.frame_number)
                    if stem in stems:raise ValueError('Output name collision')
                    if info.flag_bits & 1:raise ValueError('Encrypted selected ZIP member')
                    if info.file_size==0:raise ValueError('Empty selected JPEG member')
                    if info.compress_type not in {zipfile.ZIP_STORED,zipfile.ZIP_DEFLATED,zipfile.ZIP_BZIP2,zipfile.ZIP_LZMA}:raise ValueError('Unsupported ZIP compression')
                    seen.add(info.header_offset);stems.add(stem);image_bytes+=info.file_size
                except (ValueError,OSError) as exc:
                    errors.append({'row':number,'sequence':row.sequence,'frame_number':row.frame_number,'image_name':row.image_name,'error':type(exc).__name__,'detail':str(exc)})
    except (OSError,zipfile.BadZipFile,RuntimeError) as exc:
        errors.append({'row':'','sequence':'','frame_number':'','image_name':'','error':type(exc).__name__,'detail':'Fatal archive error: '+str(exc)})
    with (out/'preflight_errors.csv').open('w',encoding='utf-8',newline='') as dest:
        writer=csv.DictWriter(dest,fieldnames=ERROR_FIELDS);writer.writeheader();writer.writerows(errors)
    report={'builder_version':VERSION,'status':'metadata_pass' if not errors else 'failed','source':str(source),'source_bytes':source.stat().st_size,'manifest_sha256':digest(manifest),'expected_frames':len(df),'matched_frames':len(seen),'errors':len(errors),'selected_image_bytes':image_bytes,'selected_image_gib':round(image_bytes/(1024**3),3),'expected_output_files':2*len(df)+3,'by_split':df.assigned_split.value_counts().to_dict(),'by_role':df.sample_role.value_counts().to_dict(),'limitations':['ZIP metadata only: JPEG dimensions, decoding, CRC and EXIF are checked during full build.','Image bytes exclude labels, filesystem allocation overhead and report files. Allow additional free space.','No annotation-completeness or cross-sequence image-duplicate audit performed.']}
    (out/'preflight_report.json').write_text(json.dumps(report,indent=2,ensure_ascii=False),encoding='utf-8')
    print(f"{report['status'].upper()}: {len(seen)}/{len(df)} references, {len(errors)} errors; images {report['selected_image_gib']} GiB. Report: {out/'preflight_report.json'}")
    return 0 if not errors else 1

def run(args) -> int:
    source=args.source.resolve();manifest=args.manifest.resolve();out=args.output.resolve()
    if not source.is_file() or not manifest.is_file():raise ValueError('Source ZIP and manifest must be existing files')
    if out.exists() and (not out.is_dir() or any(out.iterdir())):raise ValueError('Output must be absent or an empty directory; refusing overwrite')
    df=load_manifest(manifest)
    out.mkdir(parents=True,exist_ok=True)
    if getattr(args,'check_only',False):return preflight(source,manifest,df,out)
    report={'builder_version':VERSION,'source':str(source),'source_bytes':source.stat().st_size,'source_sha256':digest(source) if args.hash_source else None,'manifest':str(manifest),'manifest_sha256':digest(manifest),'expected_frames':len(df),'dimensions':[640,512],'class_names':{'0':'thermal_target'},'status':'building','exported_frames':0,'errors':0,'by_split':{},'by_role':{},'notes':['Strict 640x512; no resizing, clipping, or silent annotation repair.','Positive boxes invalid or out of bounds are skipped, never converted to negative.','Thermal directory preferred; ambiguous matches skipped.','Only selected JPEG payloads read; ZIP central directory indexed.']}
    errors=[];counts=Counter();roles=Counter();seen_members=set();seen_output=set()
    for split in SPLITS:
        for category in ('images','labels'):(out/category/split).mkdir(parents=True,exist_ok=True)
    try:
        with zipfile.ZipFile(source,'r') as z:
            index=index_zip(z,set(df.sequence))
            for number,row in enumerate(tqdm(df.itertuples(index=False),total=len(df),desc='Build YOLO',unit='frames'),start=2):
                image_path=label_path=None
                try:
                    # Validate label semantics before opening an image, including invalid flags.
                    label=make_label(row,640,512)
                    info=resolve_member(index,row)
                    if info.header_offset in seen_members:raise ValueError('Same RAW member referenced twice')
                    raw=z.read(info)
                    with Image.open(io.BytesIO(raw)) as im:
                        if im.format!='JPEG':raise ValueError('Payload is not JPEG')
                        if im.size!=(640,512):raise ValueError(f'Expected 640x512, got {im.size}')
                        if im.mode not in ('L','RGB'):raise ValueError(f'Unsupported image mode {im.mode}')
                        if im.getexif().get(274,1)!=1:raise ValueError('Nontrivial EXIF orientation; coordinate interpretation requires review')
                        im.load() # decode completely; corrupt/truncated JPEG is an error
                    stem=output_stem(row.sequence,row.frame_number)
                    if stem in seen_output:raise ValueError('Output name collision')
                    image_path=out/'images'/row.assigned_split/(stem+'.jpg')
                    label_path=out/'labels'/row.assigned_split/(stem+'.txt')
                    with image_path.open('xb') as dest:dest.write(raw)
                    with label_path.open('x',encoding='utf-8',newline='\n') as dest:dest.write(label)
                    seen_output.add(stem);seen_members.add(info.header_offset)
                    counts[row.assigned_split]+=1;roles[row.sample_role]+=1
                except (ValueError,OSError,zipfile.BadZipFile,RuntimeError,EOFError) as exc:
                    # Only paths created by this row, under an initially empty output, are cleaned.
                    for target in (image_path,label_path):
                        if target is not None and target.is_file():target.unlink()
                    errors.append({'row':number,'sequence':row.sequence,'frame_number':row.frame_number,'image_name':row.image_name,'error':type(exc).__name__,'detail':str(exc)})
    except (OSError,zipfile.BadZipFile,RuntimeError) as exc:
        errors.append({'row':'','sequence':'','frame_number':'','image_name':'','error':type(exc).__name__,'detail':'Fatal archive error: '+str(exc)})
    with (out/'build_errors.csv').open('w',encoding='utf-8',newline='') as dest:
        writer=csv.DictWriter(dest,fieldnames=ERROR_FIELDS);writer.writeheader();writer.writerows(errors)
    yaml_data={'path':out.as_posix(),'train':'images/train','val':'images/val','test':'images/test','names':{0:'thermal_target'}}
    (out/'data.yaml').write_text(yaml.safe_dump(yaml_data,sort_keys=False,allow_unicode=True),encoding='utf-8')
    report.update(exported_frames=sum(counts.values()),errors=len(errors),by_split={s:counts[s] for s in SPLITS},by_role={r:roles[r] for r in ROLES},status='complete' if not errors and sum(counts.values())==len(df) else 'partial')
    report['file_counts']={s:{'images':len(list((out/'images'/s).glob('*.jpg'))),'labels':len(list((out/'labels'/s).glob('*.txt')))} for s in SPLITS}
    (out/'build_report.json').write_text(json.dumps(report,indent=2,ensure_ascii=False),encoding='utf-8')
    print(f"{report['status'].upper()}: {report['exported_frames']}/{len(df)} frames, {len(errors)} errors. Report: {out/'build_report.json'}")
    return 0 if report['status']=='complete' else 1

def main() -> int:
    p=argparse.ArgumentParser(description=__doc__,formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument('--source',required=True,type=Path,help='Read-only AntiUAV410_RAW.zip')
    p.add_argument('--manifest',required=True,type=Path,help='selected_frames.csv or .csv.gz')
    p.add_argument('--output',required=True,type=Path,help='New or empty output directory')
    p.add_argument('--hash-source',action='store_true',help='Optionally stream entire ZIP to compute SHA-256; adds a full-file read')
    p.add_argument('--check-only',action='store_true',help='Check all selected ZIP paths and label semantics without reading JPEG payloads; write preflight reports only. Use a separate empty output directory.')
    args=p.parse_args()
    if args.check_only and args.hash_source:p.error('--hash-source cannot be combined with metadata-only --check-only')
    try:return run(args)
    except (ValueError,OSError,KeyError) as exc:
        print(f'FATAL: {exc}',file=sys.stderr);return 2
if __name__=='__main__':raise SystemExit(main())
''')
open('/tmp/dl/team_package/selected_frames.csv.gz', 'wb').write(base64.b64decode('...'))
print('пакет команди записано')

In [ ]:
!nvidia-smi -L
!pip install -q ultralytics
%cd /kaggle/working
# ======= ДАНІ: командний датасет Anti-UAV410 SKY + синтетичне сонце =======
DRIVE_URL = 'https://drive.google.com/file/d/1zsdazmKS3mHaEZWS2BnqbYHPEcIaH5WR/view'   # AntiUAV410.zip (сирий архів)
PKG_URL = ''   # не потрібен: пакет команди (маніфест відібраних кадрів + builder) вбудовано в клітинку вище
TRAIN_EVERY = 1   # маніфест у клітинці вище вже проріджений (train: кожен 3-й кадр із ціллю, val: кожен 4-й), тут більше не проріджуємо
VAL_EVERY = 1     # val у маніфесті вже кожен 4-й кадр. Test тут не збираємо: фінальний тест робимо на 2080 SUPER з повним пакетом
SPLITS = ['train', 'val']
SUN_FRAC = 0.10                 # до train додаємо копії 10% кадрів із синтетичним сонцем (ціль лишається, сонце за 40+ px від неї)

import os, glob, re, subprocess, zipfile, shutil, random, yaml
import numpy as np, pandas as pd, cv2
DL = '/tmp/dl'; os.makedirs(DL, exist_ok=True)

def fetch(url, out):
    # Великі файли Google Drive віддає через сторінку "не вдалося перевірити на віруси".
    # Качаємо напряму з drive.usercontent.google.com з confirm=t, а якщо прийшла сторінка, беремо з неї форму і пробуємо ще раз.
    import requests
    if os.path.exists(out) and os.path.getsize(out) > 1 << 20: print('вже є', out); return
    fid = (re.search(r'/d/([\w-]{20,})', url) or re.search(r'id=([\w-]{20,})', url)).group(1)
    s, u, params = requests.Session(), 'https://drive.usercontent.google.com/download', {'id': fid, 'export': 'download', 'confirm': 't'}
    for _ in range(3):
        r = s.get(u, params=params, stream=True, timeout=120)
        if 'text/html' not in r.headers.get('Content-Type', ''):
            total, done, mark = int(r.headers.get('Content-Length', 0)), 0, 0
            with open(out, 'wb') as f:
                for chunk in r.iter_content(8 << 20):
                    f.write(chunk); done += len(chunk)
                    if done - mark > 1 << 30: mark = done; print(f'  {done / 2**30:.1f} / {total / 2**30:.1f} ГБ', flush=True)
            print(f'скачано {out}: {done / 2**30:.2f} ГБ'); return
        html = r.text
        form = re.search(r'<form[^>]+action="([^"]+)"', html)
        inputs = dict(re.findall(r'<input[^>]*name="([^"]+)"[^>]*value="([^"]*)"', html))
        if not (form and inputs): break
        u, params = form.group(1).replace('&amp;', '&'), inputs
    t = re.findall(r'<title>(.*?)</title>', html, re.S)
    raise SystemExit('Google Drive не віддав файл: ' + (t[0].strip() if t else '?') + '. Перевірте доступ "Усі, хто має посилання".')

# 1. Сирий архів і пакет команди
RAW = f'{DL}/AntiUAV410.zip'; fetch(DRIVE_URL, RAW)
assert zipfile.is_zipfile(RAW), 'DRIVE_URL має вести на zip-архів Anti-UAV410'
if not os.path.exists(f'{DL}/team_package/build_antiuav_yolo.py') and PKG_URL.startswith('http'):
    fetch(PKG_URL, f'{DL}/pkg.zip'); zipfile.ZipFile(f'{DL}/pkg.zip').extractall(DL)
PKG = os.path.dirname(glob.glob(f'{DL}/**/build_antiuav_yolo.py', recursive=True)[0])
subprocess.run('pip install -q "pandas>=2.2" "Pillow>=10" "PyYAML>=6" tqdm', shell=True)

# 2. Проріджуємо маніфест: train кожен TRAIN_EVERY-й позитив у серії, val кожен VAL_EVERY-й, негативи і test усі
m = pd.read_csv(f'{PKG}/selected_frames.csv.gz', dtype={'sequence': str, 'image_name': str, 'sample_role': str, 'assigned_split': str})
m = m[m.assigned_split.isin(SPLITS)].sort_values(['assigned_split', 'sequence', 'frame_number'])
pos = m.sample_role == 'positive'
rank = m[pos].groupby(['assigned_split', 'sequence']).cumcount()
step = m.loc[pos, 'assigned_split'].map({'train': TRAIN_EVERY, 'val': VAL_EVERY, 'test': 1})
keep = pd.Series(True, index=m.index); keep[pos] = (rank % step == 0)
m = m[keep]
MAN = f'{DL}/manifest_used.csv'; m.to_csv(MAN, index=False)
print('кадрів у збірці:', m.groupby(['assigned_split', 'sample_role']).size().to_dict())

# 3. Збірка командним builder: перевірка шляхів, потім повна збірка (кожен JPEG декодується і перевіряється)
SRC = '/tmp/sky_yolo'
if not os.path.exists(f'{SRC}/data.yaml'):
    shutil.rmtree('/tmp/sky_check', ignore_errors=True); shutil.rmtree(SRC, ignore_errors=True)
    r = subprocess.run(f'python {PKG}/build_antiuav_yolo.py --source {RAW} --manifest {MAN} --output /tmp/sky_check --check-only', shell=True)
    if r.returncode != 0:
        if os.path.exists('/tmp/sky_check/preflight_errors.csv'): print(open('/tmp/sky_check/preflight_errors.csv').read()[:3000])
        raise SystemExit('Перевірка архіву не пройшла: шляхи в архіві не збігаються з маніфестом. Можливо, на Drive інший архів, ніж у того, хто робив пакет.')
    r = subprocess.run(f'python {PKG}/build_antiuav_yolo.py --source {RAW} --manifest {MAN} --output {SRC}', shell=True)
    errs = pd.read_csv(f'{SRC}/build_errors.csv')
    print('помилок збірки:', len(errs))
    if len(errs) > 0.01 * len(m): raise SystemExit('Забагато помилок збірки, див. /tmp/sky_yolo/build_errors.csv')

# 4. Синтетичне сонце: копії частини кадрів train із пересвіченим диском і ореолом
def add_sun(img, boxes, rng):
    h, w = img.shape
    r = rng.uniform(8, 30); sig = rng.uniform(1.0, 3.0) * r
    for _ in range(200):
        if boxes and rng.random() < 0.5:                       # половина сонць поруч із ціллю (40-120 px)
            bx, by = boxes[rng.integers(len(boxes))]; ang = rng.uniform(0, 2 * np.pi); d = rng.uniform(40, 120) + r
            cx, cy = bx + d * np.cos(ang), by + d * np.sin(ang)
        else:
            cx, cy = rng.uniform(r, w - r), rng.uniform(r, h * 0.6)
        if not (r <= cx < w - r and r <= cy < h - r): continue
        if boxes and min(np.hypot(bx - cx, by - cy) for bx, by in boxes) < 40 + r: continue
        break
    yy, xx = np.mgrid[:h, :w]; d = np.hypot(xx - cx, yy - cy)
    halo = np.where(d <= r, 1.0, np.exp(-((d - r) ** 2) / (2 * sig ** 2)))
    out = img.astype(np.float32) + (255 - img.astype(np.float32)) * halo * rng.uniform(0.6, 1.0)
    out[d <= r] = 255
    return np.clip(out, 0, 255).astype(np.uint8)

if SUN_FRAC > 0 and not glob.glob(f'{SRC}/images/train/sun_*'):
    rng = np.random.default_rng(0)
    files = sorted(glob.glob(f'{SRC}/images/train/*.jpg')); random.Random(0).shuffle(files)
    for p in files[:int(len(files) * SUN_FRAC)]:
        n = os.path.basename(p)[:-4]; img = cv2.imread(p, 0); H, W = img.shape
        lab = open(f'{SRC}/labels/train/{n}.txt').read()
        centers = [(float(l.split()[1]) * W, float(l.split()[2]) * H) for l in lab.splitlines() if l.strip()]
        cv2.imwrite(f'{SRC}/images/train/sun_{n}.jpg', add_sun(img, centers, rng), [cv2.IMWRITE_JPEG_QUALITY, 95])
        open(f'{SRC}/labels/train/sun_{n}.txt', 'w').write(lab)

d = yaml.safe_load(open(f'{SRC}/data.yaml')); d['path'] = SRC
yaml.safe_dump(d, open('/kaggle/working/data.yaml', 'w'))
for s in SPLITS:
    labs = glob.glob(f'{SRC}/labels/{s}/*.txt')
    neg = sum(os.path.getsize(f) == 0 for f in labs); sun = len(glob.glob(f'{SRC}/images/{s}/sun_*'))
    print(f'{s}: {len(labs)} кадрів, негативів {neg} ({neg / max(len(labs), 1):.1%}), із сонцем {sun}')


In [ ]:
# Готовий датасет зберігаємо в Output: інші акаунти можуть додати його як Input і не качати з Drive ще раз
!cd {os.path.dirname(SRC)} && zip -qr /kaggle/working/auav_yolo.zip {os.path.basename(SRC)} && ls -la /kaggle/working/auav_yolo.zip

## Навчання
P2-голова не має готових ваг: модель будується з yaml, а старт з ваг COCO того ж розміру. Аугментації під тепловізор: без зміни кольору, помірний масштаб, щоб дрібні цілі не зникали.

In [ ]:
import sys; sys.path.insert(0, '/kaggle/working')
%cd /kaggle/working
from run_yolo import build, AUG
m, pre = build(MODEL)
m.train(data='/kaggle/working/data.yaml', pretrained=pre, epochs=EPOCHS, time=HOURS, imgsz=640, batch=BATCH, device=GPUS,
        project='/kaggle/working/runs', name=MODEL, exist_ok=True, patience=15, cos_lr=True, workers=4, save_period=5, **AUG)

## Оцінка за ТЗ на val
Поріг впевненості підбирається так, щоб FP на кадр ≤ 0.7. Recall лише для цілей 5-50 px, влучання = центр рамки в рамці цілі ±3 px. Швидкість тут на T4 у PyTorch; фінальний FPS міряємо на 2080 SUPER через TensorRT.

In [ ]:
import glob, shutil, json, subprocess
best = sorted(glob.glob(f'/kaggle/working/runs/**/{MODEL}/weights/best.pt', recursive=True))[-1]; print(best)
shutil.copy(best, '/kaggle/working/best.pt')
shutil.copy(os.path.join(os.path.dirname(os.path.dirname(best)), 'results.csv'), '/kaggle/working/results.csv')
!python run_yolo.py eval --weights /kaggle/working/best.pt --val {SRC}:val --test {SRC}:val --json /kaggle/working/metrics.json | tail -30
from ultralytics import YOLO
YOLO('/kaggle/working/best.pt').export(format='onnx', imgsz=[512, 640], simplify=True)
r = json.load(open('/kaggle/working/metrics.json'))
print(MODEL, 'conf', r['conf'], 'topk', r['topk'], 'val', r['val'], 'T4 PyTorch', r['speed'])